## Tratamiento de datos para la regresión.

Para el entrenamiento de los modelos de regresión necesitamos obtener variables dependientes de las ventas por fecha. En este caso diarias. El resto de datos no aportan demasiado valor. Por lo tanto vamos a realizar lo siguiente:

- Cargar el csv ya limpio para la regresión.

- Prescindir de las variables que no sean necesarias.

- Tipar correctamente las columnas que sean necesarias.

- Generar nuevas variables que aporten información sobre las ventas diarias.

#### Librerías utilizadas.

In [193]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#### Carga del csv limpio y visualización.

In [194]:
df = pd.read_csv('../output/data_limpio_regresion.csv')

Visualizamos los datos para comprobar que están correctos y su formato.

In [195]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 476888 entries, 0 to 476887
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    476888 non-null  int64  
 1   StockCode    476888 non-null  str    
 2   Description  476888 non-null  str    
 3   Quantity     476888 non-null  int64  
 4   InvoiceDate  476888 non-null  str    
 5   UnitPrice    476888 non-null  float64
 6   CustomerID   355996 non-null  float64
 7   Country      476888 non-null  str    
dtypes: float64(2), int64(2), str(4)
memory usage: 29.1 MB


In [196]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED RETROSPOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,22745,POPPY'S PLAYHOUSE BEDROOM,6,12/1/2010 8:34,2.10,13047.0,United Kingdom


#### Variables necesarias.

Determinamos que las columnas necesarias para entrenar los modelos de regresión son las siguientes.

- InvoiceDate: Para agrupar los valores por día.

- Quantity: Apoyo para obtener las ganancias totales por día.

- UnitPrice: Apoyo para obtener las ganancias totales por día.

Por lo tanto eliminamos el resto de columnas para que no ensucien el nuevo dataset.

In [197]:
df_transformado = df.copy()
df_transformado = df_transformado[['InvoiceDate', 'Quantity', 'UnitPrice']]
df_transformado.head(5)

,InvoiceDate,Quantity,UnitPrice
0,12/1/2010 8:26,6,2.55
1,12/1/2010 8:26,6,3.39
2,12/1/2010 8:26,8,2.75
3,12/1/2010 8:26,6,3.39
4,12/1/2010 8:26,6,3.39


#### Tipado de las columnas.

Tanto Quantity como UnitPrice están bien tipadas. Pero el caso de InvoiceDate es diferente ya que está en formato String. Para poder trabajar correctamente pasamos esta columna a formato Fecha.

In [198]:
df_transformado['InvoiceDate'] = pd.to_datetime(df_transformado['InvoiceDate'], format='%m/%d/%Y %H:%M')
df_transformado.info()
df_transformado.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 476888 entries, 0 to 476887
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceDate  476888 non-null  datetime64[us]
 1   Quantity     476888 non-null  int64         
 2   UnitPrice    476888 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 10.9 MB


,InvoiceDate,Quantity,UnitPrice
0,2010-12-01 08:26:00,6,2.55
1,2010-12-01 08:26:00,6,3.39
2,2010-12-01 08:26:00,8,2.75
3,2010-12-01 08:26:00,6,3.39
4,2010-12-01 08:26:00,6,3.39


#### Creación de nuevas variables.

Esta parte es fundamental para entrenar al modelo de la mejor forma posible. Tenemos que representar los siguienes patrones con nuevas variables:

- Ganancias por día de la semana. Como sólo tenemos datos de un año es irrelevante tomar como referencia el día del año. El día de la semana se repite en muchas ocasiones y puede dar un aproximado real.

- Ganancias por día del mes. Al igual que con las ganancias por día de la semana representa mejor las tendencias dentro de cada mes para intentar aproximar valores.

- El mes en el que se produce las ventas de ese día, para capturar patrones estacionarios.

Primero generamos un nuevo dataframe agrupado por fechas y añadimos una variable de ganancias totales.

In [199]:
df_agrupado = df_transformado.groupby('InvoiceDate').agg({'Quantity': 'sum', 'UnitPrice': 'mean'}).reset_index()
df_agrupado['GananciasTotales'] = (df_agrupado['Quantity'] * df_agrupado['UnitPrice']).round(2)
df_agrupado.head(5)

,InvoiceDate,Quantity,UnitPrice,GananciasTotales
0,2010-12-01 08:26:00,40,3.910000,156.40
1,2010-12-01 08:28:00,12,1.850000,22.20
2,2010-12-01 08:34:00,66,5.043333,332.86
3,2010-12-01 08:35:00,3,5.950000,17.85
4,2010-12-01 08:45:00,362,2.105294,762.12


Ahora toca compactar todas las ganancias en días únicos.

In [200]:
df_transformado['Fecha'] = df_transformado['InvoiceDate'].dt.date
df_agrupado = df_transformado.groupby('Fecha').agg({'Quantity': 'sum', 'UnitPrice': 'mean'}).reset_index()
df_agrupado['GananciasTotales'] = (df_agrupado['Quantity'] * df_agrupado['UnitPrice']).round(2)
df_agrupado.head(5)

,Fecha,Quantity,UnitPrice,GananciasTotales
0,2010-12-01,13407,3.233640,43353.42
1,2010-12-02,12939,2.881431,37282.83
2,2010-12-03,9319,3.484525,32472.29
3,2010-12-05,11106,2.659746,29539.14
4,2010-12-06,14508,3.283226,47633.05


Para mejorar los datos, vamos a crear nuevas variables que indiquen el día de la semana, el día del mes y el propio mes.

In [201]:
df_agrupado['DiaSemana'] = pd.to_datetime(df_agrupado['Fecha']).dt.dayofweek  # 0=Lunes, 6=Domingo
df_agrupado['DiaMes'] = pd.to_datetime(df_agrupado['Fecha']).dt.day
df_agrupado['Mes'] = pd.to_datetime(df_agrupado['Fecha']).dt.month
df_agrupado.head()

,Fecha,Quantity,UnitPrice,GananciasTotales,DiaSemana,DiaMes,Mes
0,2010-12-01,13407,3.233640,43353.42,2,1,12
1,2010-12-02,12939,2.881431,37282.83,3,2,12
2,2010-12-03,9319,3.484525,32472.29,4,3,12
3,2010-12-05,11106,2.659746,29539.14,6,5,12
4,2010-12-06,14508,3.283226,47633.05,0,6,12


Borramos los datos que ya no necesitamos como el Quatity y UnitPrice, y convertimos en variables categóricas dia de la semana y mes.

In [202]:
df_agrupado = df_agrupado.drop(columns=['Fecha', 'Quantity', 'UnitPrice'])

dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

df_agrupado['DiaSemana'] = df_agrupado['DiaSemana'].apply(lambda x: dias_semana[x])
df_agrupado['Mes'] = df_agrupado['Mes'].apply(lambda x: meses[x-1])
df_agrupado.head(5)

,GananciasTotales,DiaSemana,DiaMes,Mes
0,43353.42,Miércoles,1,Diciembre
1,37282.83,Jueves,2,Diciembre
2,32472.29,Viernes,3,Diciembre
3,29539.14,Domingo,5,Diciembre
4,47633.05,Lunes,6,Diciembre


#### Comprobación de los nuevos valores.

Visualizamos las características de ganancias totales para comprobar que no existen valores extraños.

In [203]:
df_agrupado.describe()

,GananciasTotales,DiaMes
count,305.00000,305.000000
mean,27026.22177,15.137705
std,11687.88560,8.649273
min,3513.17000,1.000000
25%,19392.44000,8.000000
50%,25406.29000,15.000000
75%,32832.32000,22.000000
max,73454.82000,31.000000


Para reducir un poco los picos máximos y mínimos realizamos una normalización de los datos.

In [204]:
scaler = StandardScaler()

columnas_numericas = ['GananciasTotales']
df_agrupado[columnas_numericas] = scaler.fit_transform(df_agrupado[columnas_numericas])
df_agrupado.head()

,GananciasTotales,DiaSemana,DiaMes,Mes
0,1.399229,Miércoles,1,Diciembre
1,0.878984,Jueves,2,Diciembre
2,0.466724,Viernes,3,Diciembre
3,0.215355,Domingo,5,Diciembre
4,1.765990,Lunes,6,Diciembre


#### Preprocesado de variables categóricas.

Para el correcto tratamiento de los días de la semana, meses e incluso días del mes (variable categóricas), es necesario aplicar una técnica de preprocesado como OneHotEncoding, que deja los valores en formato de bits y evita que tome con mayor importancia unas variables de otras.

In [205]:
encoder = OneHotEncoder(sparse_output=False, drop='first')
columnas_categoricas = ['DiaSemana', 'DiaMes', 'Mes']

df_encoded = pd.DataFrame(encoder.fit_transform(df_agrupado[columnas_categoricas]))
df_encoded.columns = encoder.get_feature_names_out(columnas_categoricas)

df_final = pd.concat([df_agrupado.drop(columns=columnas_categoricas), df_encoded], axis=1)

df_final.head()

,GananciasTotales,DiaSemana_Jueves,DiaSemana_Lunes,DiaSemana_Martes,DiaSemana_Miércoles,DiaSemana_Viernes,DiaMes_2,DiaMes_3,DiaMes_4,DiaMes_5,...,Mes_Diciembre,Mes_Enero,Mes_Febrero,Mes_Julio,Mes_Junio,Mes_Marzo,Mes_Mayo,Mes_Noviembre,Mes_Octubre,Mes_Septiembre
0,1.399229,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.878984,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.466724,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.215355,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.765990,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Exportar el dataframe transformado.

Para la posterior utilización del dataframe en los modelos de regresión lineal lo exportamos a la misma carpeta que con la limpieza.

In [206]:
import os
os.makedirs('../output', exist_ok=True)

df_final.to_csv('../output/data_transformado_regresion.csv', index=False, encoding='ISO-8859-1')
print(f"Dataset de regresión exportado: {len(df_final)} filas")
print("Archivo: ../output/data_transformado_regresion.csv")

Dataset de regresión exportado: 305 filas
Archivo: ../output/data_transformado_regresion.csv
